# Huggingface 로그인

In [ ]:
# 신청이 승인되면 계정 settings 페이지에서 access token 복사하여 사용
huggingface-cli login

# 패키지 설치

In [ ]:
pip install transformers
pip install chromadb
pip install pdfplumber
pip install torch

# import

In [ ]:
import re
import fitz
import random

import numpy as np
import pandas as pd

import chromadb
import pdfplumber

import torch
import transformers
from transformers import AutoTokenizer, AutoModel

# 시드 고정

In [ ]:
def seed_everything(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if use multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)

seed_everything(42)

# DB 연결

In [ ]:
client = chromadb.Client()
word_test = client.create_collection(name="word_test")

# 모델 로드

In [ ]:
# 임베딩 모델 로드
tokenizer = AutoTokenizer.from_pretrained("bespin-global/klue-sroberta-base-continue-learning-by-mnr")
model = AutoModel.from_pretrained("bespin-global/klue-sroberta-base-continue-learning-by-mnr")

# LLM 모델 로드
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

pipeline = transformers.pipeline("text-generation",
                                 model=model_id,
                                 model_kwargs={"torch_dtype": torch.bfloat16},
                                 device_map="auto")

# PDF 파싱 및 전처리

In [ ]:
def parse_doc(file_path):
    table_li =[]
    table_names = []
    table_dict = dict()
    txt_pages = []
    txt_contents = []
    pattern = re.compile(r'표\d+\.')

    # PDF 열기
    with pdfplumber.open(file_path) as pdf:
        for i, page in enumerate(pdf.pages[4:]):
            # 페이지에서 테이블, 텍스트 추출
            text = page.extract_text()
            text = text.replace('사용자_매뉴얼\nv1.0.0\n2024.03.27\n', '').replace('사용자_매뉴얼 v1.0.0 2024.03.27', '')
            sentences = text.split('\n')
            tables = page.extract_tables()

            # 패턴을 포함하는 문장 찾기
            for sentence in sentences:
                if tables and pattern.search(sentence):
                    table_names.append(sentence)
                if sentence:
                    txt_pages.append(i+5)
                    txt_contents.append(sentence)
            for table in tables:
                df = pd.DataFrame(table[1:], columns=table[0])
                markdown_table = df.to_markdown(index=False)
                table_li.append(markdown_table)

        for table, name in zip(table_li,table_names):
            table_dict[name]=table
    
    return txt_pages, txt_contents, table_dict

In [ ]:
# 4depth 기준으로 같은 문단인 문장 합치기
def merge_text_data(pages, contents):
    pattern = re.compile(r'\d{1,}-\d{1,}-\d{1,}-\d{1,}\s') # 숫자-숫자-숫자-숫자빈칸 형식의 정규표현식
    start_idx = 0
    indices = []
    merged_lists = []
    merged_pages = []
    
    for idx, elem in enumerate(contents):
        if isinstance(elem, str): # 요소가 문자열인지 확인
            if pattern.match(elem): # 정규표현식과 매치되는지 확인
                indices.append(idx) # 매치되는 경우 인덱스 추가

    # indices를 기준으로 문장 병합
    for i, idx in enumerate(indices):
        if i == len(indices)-1: # 마지막까지 포함시키기 위함
            merged_text = ' '.join(contents[start_idx:idx])
            merged_page = pages[start_idx:idx]
            merged_lists.append(merged_text)
            merged_pages.append(str(list(set(merged_page))))
            merged_text = ' '.join(contents[idx:len(contents)])
            merged_page = merged_page + pages[idx:len(pages)]
        else:
            merged_text = ' '.join(contents[start_idx:idx])
            merged_page = pages[start_idx:idx]
            
        merged_lists.append(merged_text)
        merged_pages.append(str(list(set(merged_page))))
        start_idx = idx
            
    return merged_lists, merged_pages

# Text chunk를 임베딩하여 Vector DB에 저장

In [ ]:
# db에 임베딩 저장
def make_txt_emd(sentences):
    encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    attention_mask = encoded_input['attention_mask']
        
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_result = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    return [tensor.tolist() for tensor in sum_result]

In [ ]:
def save_doc(merge_list, collection, pages):
    txt_emd_li = make_txt_emd(merge_list)
    
    # id list 생성
    id_list=[]
    for idx in range(len(txt_emd_li)):
        id_list.append(f'Sentence_{idx}')
    
    # DB에 데이터 추가
    metadatas = []
    for page in pages:
        metadatas.append({'page':page})
        
    collection.add(embeddings= txt_emd_li, # text 임베딩 리스트
                   documents=merge_list, # text 리스트
                   metadatas=metadatas,
                   ids=id_list)

# 쿼리 및 LLM 모델 답변 생성

In [ ]:
# LLM에 prompt 입력하여 답변 생성
def make_answer(context, query):
    messages = [{"role": "system", "content": context},
                {"role": "user", "content": query}]
    prompt = pipeline.tokenizer.apply_chat_template(messages,
                                                    tokenize=False,
                                                    add_generation_prompt=True)
    
    terminators = [pipeline.tokenizer.eos_token_id,
                   pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")]
    
    outputs = pipeline(prompt,
                       max_new_tokens=512,
                       eos_token_id=terminators,
                       pad_token_id=128001,
                       do_sample=True,
                       temperature=0.2, #다음에 올 단어를 선택할 때 불확실성을 조절. 값이 낮을수록 생성된 텍스트는 더 일관성있는 내용일 수 있음
                       top_p=0.9)

    return outputs[0]["generated_text"][len(prompt):]

In [ ]:
def q_and_a(question,collection):
    # 질문 임베딩
    q_emd = make_txt_emd([question])
    # DB 쿼리
    result = collection.query(query_embeddings=q_emd[0],
                              n_results=2)
    # prompt생성
    context = result['documents'][0]
    metadatas = result['metadatas'][0]
    page = (metadatas[0]['page'][1:-1] + ', ' + metadatas[1]['page'][1:-1])
    query = question + ". 답변에 다른 설명은 덧붙이지 말고, 제공된 정보에 있는 내용에서 질문에 대한 답변만 해주고, 정보에 없는 부분은 알 수 없다고 답해줘"
    messages = [{"role": "system", "content": context},
                {"role": "user", "content": query},]
    
    # 답변 생성
    print(make_answer(context, query) + f'\n인용된 페이지: {page}')

# 결과

In [ ]:
# PDF 파일경로
file_path = "사용자_매뉴얼.pdf"

# 문서 파싱 및 전처리
txt_pages, txt_contents, table_dict = parse_doc(file_path)
merged_lists, merged_pages = merge_text_data(txt_pages,txt_contents)

# 임베딩 및 DB에 저장
save_doc(merged_lists, word_test, merged_pages)

# 쿼리 및 LLM 답변 출력
question = "매뉴얼을 다운로드 할 수 있는 명령어가 무엇인지 알려줘"
q_and_a(question, word_test)